# dist-send-recv-pair — faded example 2: Broadcast by looping dist.send to every non-self rank

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dist-send-recv-pair`. Running the beacon reports progress on the `Distributed: dist.send/recv pair` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: dist.send/recv pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dist-send-recv-pair`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dist-send-recv-pair"
DD_SUBTOPIC = "Distributed: dist.send/recv pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Broadcasting via point-to-point primitives requires the source rank to send its tensor to every other rank individually. The pattern is a `for` loop over all ranks, skipping self. Each iteration calls `dist.send(tensor, dst=other)`, and the corresponding `dist.recv` on each recipient completes the matched pair.

## Faded exercise 2

Complete the sender branch of `broadcast_send`. When `rank == src`, loop over all ranks and send to those that are not `src`.

**Fill in:** A `for` loop over `range(world_size)` that calls `dist_module.send(tensor, dst=other)` for every `other != src`.

In [ ]:
def broadcast_send(rank, world_size, dist_module, src, tensor):
    """Sender side of a manual broadcast. Returns None."""
    if rank == src:
        raise NotImplementedError()  # TODO: A `for` loop over `range(world_size)` that calls `dist_module.send(tensor, dst=other)` for every `other != src`.
    else:
        import torch
        buf = torch.zeros_like(tensor)
        dist_module.recv(buf, src=src)
        tensor.copy_(buf)


def _test():
    import torch
    from unittest.mock import MagicMock, call

    dist_mock = MagicMock()
    tensor = torch.tensor([1.0, 2.0, 3.0])
    world_size = 4
    src = 0

    broadcast_send(rank=src, world_size=world_size, dist_module=dist_mock,
                   src=src, tensor=tensor)

    # Should have sent to ranks 1, 2, 3 (all non-src)
    expected_calls = [call(tensor, dst=1), call(tensor, dst=2), call(tensor, dst=3)]
    assert dist_mock.send.call_count == world_size - 1
    for c in expected_calls:
        assert c in dist_mock.send.call_args_list
    # Should NOT have sent to itself
    assert call(tensor, dst=src) not in dist_mock.send.call_args_list


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def broadcast_send(rank, world_size, dist_module, src, tensor):
    """Sender side of a manual broadcast. Returns None."""
    if rank == src:
        for other in range(world_size):
            if other != src:
                dist_module.send(tensor, dst=other)
    else:
        import torch
        buf = torch.zeros_like(tensor)
        dist_module.recv(buf, src=src)
        tensor.copy_(buf)
```
</details>